In [ ]:
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
import random

sys.path.append('../')
from data_provider.data_loader import load_dataset


#### Experiment Configuration

In [4]:
# Data and Model Configurations
DATA_FOLDER = '../data'
DATASET_NAME = 'train_v2_drcat_02'

# Choose a pre-trained BERT model
MODEL_NAME = 'bert-base-uncased'

# Data Preprocessing
TEST_SIZE = 0.2
VAL_SIZE = 0.1
RANDOM_SEED = 2026
MAX_LENGTH = 256

# Model Hyperparameters
WEIGHT_DECAY = 0.01

# Trainin and Optimization
BATCH_SIZE = 16
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1

# Early Stopping
PATIENCE = 2
MIN_DELTA = 0.001

# Reproducibility
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# Device Configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE


device(type='cuda')

#### Load and Split Dataset

In [5]:
df = load_dataset(DATA_FOLDER, DATASET_NAME)

# Check the column names to ensure features and labels are loaded correctly
print(df.head())

                                                text  label
0  Phones\n\nModern humans today are always on th...      0
1  This essay will explain if drivers should or s...      0
2  Driving while the use of cellular devices\n\nT...      0
3  Phones & Driving\n\nDrivers should not be able...      0
4  Cell Phone Operation While Driving\n\nThe abil...      0


#### Split Dataset

In [6]:
# Dataset is not randomly ordered, so we will shuffle it to ensure a representative split between training and testing sets

train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=df['label'],
    shuffle=True,
)

train_df, val_df = train_test_split(
    train_df,
    test_size=(VAL_SIZE / (1.0 - TEST_SIZE)),
    random_state=RANDOM_SEED,
    stratify=train_df['label'],
    shuffle=True,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f'Train set size: {len(train_df)}')
print(f'Validation set size: {len(val_df)}')
print(f'Test set size: {len(test_df)}')


Train set size: 31407
Validation set size: 4487
Test set size: 8974


#### Tokenizer and DataLoaders

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TransformerDataset(Dataset):
    def __init__(self, df, tokenizer, max_length):
        self.texts = df['text'].tolist()
        self.labels = df['label'].astype(int).tolist()
        self.encodings = tokenizer(self.texts, truncation=True, padding='max_length', max_length=max_length)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            'input_ids': torch.tensor(self.encodings['input_ids'][idx], dtype=torch.long),
            'attention_mask': torch.tensor(self.encodings['attention_mask'][idx], dtype=torch.long),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long),
        }

        return item


train_dataset = TransformerDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = TransformerDataset(val_df, tokenizer, MAX_LENGTH)
test_dataset = TransformerDataset(test_df, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

#### Model Setup

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_training_steps = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_training_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_training_steps,)

model


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

#### Training and Evaluation

In [9]:
def run_epoch(model, data_loader, optimizer=None, scheduler=None):
    is_training = optimizer is not None
    model.train() if is_training else model.eval()

    total_loss = 0.0
    y_true = []
    y_pred = []

    def process_batch(batch):
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        logits = outputs.logits

        if is_training:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            if scheduler is not None:
                scheduler.step()

        return loss, logits, labels

    if is_training:
        for batch in tqdm(data_loader):
            loss_val, logits, labels = process_batch(batch)

            total_loss += loss_val.item() * batch['input_ids'].size(0)
            predictions = torch.argmax(logits, dim=1)

            y_true.extend(labels.detach().cpu().numpy())
            y_pred.extend(predictions.detach().cpu().numpy())
    else:
        with torch.no_grad():
            for batch in tqdm(data_loader):
                loss_val, logits, labels = process_batch(batch)

                total_loss += loss_val.item() * batch['input_ids'].size(0)
                predictions = torch.argmax(logits, dim=1)

                y_true.extend(labels.detach().cpu().numpy())
                y_pred.extend(predictions.detach().cpu().numpy())

    average_loss = total_loss / len(data_loader.dataset)
    accuracy = accuracy_score(y_true, y_pred)

    return average_loss, accuracy, y_true, y_pred


best_val_accuracy = 0.0
best_model_state = None
epochs_without_improvement = 0

for epoch in range(NUM_EPOCHS):
    train_loss, train_accuracy, train_true, train_pred = run_epoch(model, train_loader, optimizer, scheduler)
    val_loss, val_accuracy, val_true, val_pred = run_epoch(model, val_loader)

    print(
        f'Epoch {epoch + 1}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_accuracy:.4f} | '
        f'Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.4f}'
    )

    if val_accuracy > best_val_accuracy + MIN_DELTA:
        best_val_accuracy = val_accuracy
        best_model_state = {
            key: value.detach().cpu().clone() for key, value in model.state_dict().items()
        }
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        break

if best_model_state is not None:
    model.load_state_dict(best_model_state)

test_loss, test_accuracy, y_true, y_pred = run_epoch(model, test_loader)
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_accuracy:.4f}')
print(classification_report(y_true, y_pred, target_names=['Human', 'AI']))


100%|██████████| 281/281 [00:13<00:00, 20.20it/s]


Epoch 1/3 | Train Loss: 0.0860 | Train Acc: 0.9688 | Val Loss: 0.0328 | Val Acc: 0.9875


100%|██████████| 281/281 [00:13<00:00, 20.23it/s]


Epoch 2/3 | Train Loss: 0.0142 | Train Acc: 0.9961 | Val Loss: 0.0216 | Val Acc: 0.9915


100%|██████████| 281/281 [00:13<00:00, 20.22it/s]


Epoch 3/3 | Train Loss: 0.0045 | Train Acc: 0.9990 | Val Loss: 0.0307 | Val Acc: 0.9913


100%|██████████| 561/561 [00:27<00:00, 20.18it/s]

Test Loss: 0.0230
Test Accuracy: 0.9920
              precision    recall  f1-score   support

       Human       1.00      0.99      0.99      5474
          AI       0.98      1.00      0.99      3500

    accuracy                           0.99      8974
   macro avg       0.99      0.99      0.99      8974
weighted avg       0.99      0.99      0.99      8974



#### Custom Text Prediction

In [10]:
def predict_custom_texts(model, tokenizer, texts, max_length):
    if isinstance(texts, str):
        texts = [texts]

    encodings = tokenizer(
        texts,
        truncation=True,
        padding='max_length',
        max_length=max_length,
        return_tensors='pt',
    )

    input_ids = encodings['input_ids'].to(DEVICE)
    attention_mask = encodings['attention_mask'].to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        probabilities = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()

    predictions = (probabilities >= 0.5).astype(int)
    results = pd.DataFrame(
        {
            'text': texts,
            'predicted_label': ['AI' if pred == 1 else 'Human' for pred in predictions],
            'ai_probability': probabilities,
        }
    )
    print(results)

    return results


#### Test BERT with Custom Inputs

In [11]:
custom_texts = [
    'This essay argues that social media can improve civic participation when paired with strong moderation policies.',
    'In conclusion, the provided evidence clearly demonstrates a multifaceted and highly structured perspective on the topic.',
    "This isn't working very well, as it seems to just be predicting AI for longer messages and Human for shorter messages.",
]

predict_custom_texts(model, tokenizer, custom_texts, MAX_LENGTH)


                                                text predicted_label  \
0  This essay argues that social media can improv...              AI   
1  In conclusion, the provided evidence clearly d...              AI   
2  This isn't working very well, as it seems to j...              AI   

   ai_probability  
0        0.999756  
1        0.999483  
2        0.986692  


,text,predicted_label,ai_probability
0,This essay argues that social media can improv...,AI,0.999756
1,"In conclusion, the provided evidence clearly d...",AI,0.999483
2,"This isn't working very well, as it seems to j...",AI,0.986692
